In [29]:
import numpy as np
import pandas as pd
import xarray as xr
from tqdm import tqdm

In [30]:
# Load agridata
df = pd.read_csv("data/raw/csv/agridata_csv_202110311352.csv")

# Convert date to datetime (handles mixed formats)
df['date'] = pd.to_datetime(df['date'], format='mixed', dayfirst=True)

# Remove rows with missing commodity names
df = df.dropna(subset=['commodity_name'])

# Get available crops
crops = sorted(df['commodity_name'].unique())
print("Available crops:", crops[:20])  # Show first 20

# ========== SELECT CROP HERE ==========
CROP = crops[0]  # Change this to select different crop
# ======================================

print(f"\nSelected crop: {CROP}")

# Filter for selected crop
df_crop = df[df['commodity_name'] == CROP].copy()

print(f"Found {len(df_crop)} records for {CROP}")

# Use modal_price as the main price
df_crop = df_crop[['state', 'market', 'modal_price', 'date']].copy()
df_crop = df_crop.dropna(subset=['modal_price'])

df_crop['mandi_state'] = df_crop['market'] + " (" + df_crop['state'] + ")"
df_crop = df_crop.drop(['market', 'state'], axis=1)

# Pivot to get prices by mandi and date
prices = df_crop.pivot_table(
    index='date', 
    columns='mandi_state', 
    values='modal_price', 
    aggfunc='first'
)

# Sort by date and remove mandis with too many missing values
prices = prices.sort_index()
prices = prices.dropna(axis=1, thresh=len(prices) * 0.5)  # Keep mandis with 50%+ data

mandis = prices.columns
T = len(prices)
N = len(mandis)

Y = prices.values.astype(float)
dates = prices.index

print(f"\nData shape: {Y.shape} (Time: {T}, Mandis: {N})")
print(f"Date range: {dates[0].date()} to {dates[-1].date()}")
print(f"Missing values: {np.isnan(Y).sum()} ({100*np.isnan(Y).sum()/(T*N):.1f}%)")

Available crops: ['Ajwan', 'Alasande Gram', 'Alasande+Gram', 'Almond(Badam)', 'Alsandikai', 'Amaranthus', 'Ambada+Seed', 'Amla(Nelli Kai)', 'Amla(Nelli+Kai)', 'Amphophalus', 'Antawala', 'Anthorium', 'Apple', 'Arecanut(Betelnut/Supari)', 'Arhar (Tur/Red Gram)(Whole)', 'Arhar+Dal(Tur+Dal)', 'Ashgourd', 'Avare+Dal', 'Balekai', 'Bamboo']

Selected crop: Ajwan
Found 706 records for Ajwan

Data shape: (244, 2) (Time: 244, Mandis: 2)
Date range: 2019-05-22 to 2021-10-30
Missing values: 167 (34.2%)


In [31]:
# Load and prepare weather data (optional)
ds = None
try:
    ds = xr.open_dataset("weather_daily.nc")
    
    # Ensure datetime compatibility
    ds["date"] = pd.to_datetime(ds["date"].values)
    
    # Ensure same ordering as mandis
    ds = ds.sel(mandi=mandis)
    
    print("Weather data loaded successfully")
except Exception as e:
    print(f"Note: Weather data not available ({e})")
    print("Continuing with prices only")

Note: Weather data not available ([Errno 2] No such file or directory: 'c:\\Users\\nisch\\Desktop\\RESEARCH\\PRICE_PREDICTION_WORK\\weather_daily.nc')
Continuing with prices only


In [32]:
# Forward fill missing values within each mandi (handle temporal gaps)
Y_filled = np.copy(Y)
for i in range(N):
    mask = ~np.isnan(Y[:, i])
    if mask.sum() > 1:
        valid_idx = np.where(mask)[0]
        valid_vals = Y[mask, i]
        # Linear interpolation for missing values
        Y_filled[:, i] = np.interp(np.arange(T), valid_idx, valid_vals, left=np.nan, right=np.nan)

# Replace remaining NaNs with column mean
for i in range(N):
    col_mean = np.nanmean(Y_filled[:, i])
    Y_filled[np.isnan(Y_filled[:, i]), i] = col_mean

# Normalize prices (zero mean, unit variance per mandi)
Y_mean = np.mean(Y_filled, axis=0)
Y_std = np.std(Y_filled, axis=0) + 1e-6  # Avoid division by zero
Y_norm = (Y_filled - Y_mean) / Y_std

# Train/validation split
split = int(0.8 * T)
train_idx = np.arange(split)
valid_idx = np.arange(split, T)

Y_train = Y_norm[:split]
Y_valid = Y_norm[split:]

train_dates = dates[:split]
valid_dates = dates[split:]

print(f"Training set: {Y_train.shape} ({len(train_dates)} days)")
print(f"Validation set: {Y_valid.shape} ({len(valid_dates)} days)")

Training set: (195, 2) (195 days)
Validation set: (49, 2) (49 days)


In [33]:
class EnKFPriceModel:
    """
    Advanced Ensemble Kalman Filter with regularization for price forecasting
    """
    def __init__(self, n_ens=120, process_noise=0.004, obs_noise=0.008, 
                 alpha_trend=0.95, inflation=1.01, damping=0.7):
        self.n_ens = n_ens
        self.process_noise = process_noise
        self.obs_noise = obs_noise
        self.alpha_trend = alpha_trend
        self.inflation = inflation
        self.damping = damping  # Damping factor to reduce innovation magnitude
        
    def initialize_ensemble(self, Y):
        """Smart initialization with reduced variance"""
        T, N = Y.shape
        X = np.zeros((N, self.n_ens))
        
        # Use recent data with trend
        recent_idx = max(0, T - 50)
        sample_data = Y[recent_idx:]
        
        mean_state = np.nanmean(sample_data, axis=0)
        std_state = np.nanstd(sample_data, axis=0)
        std_state[std_state == 0] = 0.05  # Lower default std
        
        for k in range(self.n_ens):
            # Smaller perturbations
            X[:, k] = mean_state + np.random.normal(0, std_state * 0.2, N)
        
        return X
    
    def forecast(self, Y, verbose=True):
        """Enhanced EnKF with adaptive damping"""
        T, N = Y.shape
        X = self.initialize_ensemble(Y)
        
        X_mean_out = np.zeros((T, N))
        X_std_out = np.zeros((T, N))
        
        # Track convergence
        prev_std = np.inf
        
        iterator = tqdm(range(T)) if verbose else range(T)
        
        for t in iterator:
            # FORECAST: Strong persistence with small noise
            X_prev = X.copy()
            
            for k in range(self.n_ens):
                # Strong AR process (persistence)
                X[:, k] = self.alpha_trend * X[:, k]
                
                # Very small additive noise
                noise = np.random.normal(0, np.sqrt(self.process_noise) * 0.5, N)
                X[:, k] += noise
            
            # Adaptive inflation based on ensemble spread
            X_mean = X.mean(axis=1, keepdims=True)
            X_spread = X.std(axis=1, keepdims=True)
            
            # Only inflate if ensemble collapses
            if X_spread.mean() < 0.05:
                X = self.inflation * (X - X_mean) + X_mean
            
            # UPDATE: Selective assimilation with damping
            for i in range(N):
                if not np.isnan(Y[t, i]):
                    obs = Y[t, i]
                    
                    ensemble_i = X[i, :]
                    pred_mean = ensemble_i.mean()
                    pred_var = ensemble_i.var()
                    
                    if pred_var > 1e-12:
                        # Conservative Kalman gain
                        K = pred_var / (pred_var + self.obs_noise)
                        
                        # Limit update magnitude
                        innovation = obs - pred_mean
                        max_innovation = 3 * np.sqrt(self.obs_noise)
                        innovation = np.clip(innovation, -max_innovation, max_innovation)
                        
                        # Damped update
                        update = self.damping * K * innovation
                        
                        X[i, :] += update
            
            # Outlier removal: clip extreme values
            X_mean = X.mean(axis=1)
            X_std = X.std(axis=1) + 1e-8
            for i in range(N):
                z_scores = np.abs((X[i, :] - X_mean[i]) / X_std[i])
                outlier_mask = z_scores > 3
                X[i, outlier_mask] = X_mean[i] + np.sign(X[i, outlier_mask] - X_mean[i]) * X_std[i]
            
            # Save outputs
            X_mean_out[t] = X.mean(axis=1)
            X_std_out[t] = X.std(axis=1)
        
        return X_mean_out, X_std_out

In [34]:
# Intensive optimization to minimize RMSE below 1.0
print("=" * 70)
print("INTENSIVE OPTIMIZATION (Target: RMSE < 1.0)")
print("=" * 70)

def compute_rmse(Y_true, Y_pred):
    """Compute RMSE"""
    mask = ~np.isnan(Y_true)
    if mask.sum() == 0:
        return np.inf
    Y_true_clean = Y_true[mask]
    Y_pred_clean = Y_pred[mask]
    return np.sqrt(np.mean((Y_true_clean - Y_pred_clean) ** 2))

best_rmse_global = np.inf
best_params_global = {}
iteration = 0

# Comprehensive parameter search - focus on reducing noise and trend
aggressive_params = [
    {'n_ens': 150, 'pn': 0.002, 'on': 0.005, 'at': 0.97, 'inf': 1.00, 'damp': 0.8},
    {'n_ens': 130, 'pn': 0.001, 'on': 0.003, 'at': 0.96, 'inf': 1.00, 'damp': 0.9},
    {'n_ens': 140, 'pn': 0.003, 'on': 0.006, 'at': 0.95, 'inf': 1.01, 'damp': 0.7},
    {'n_ens': 120, 'pn': 0.002, 'on': 0.004, 'at': 0.98, 'inf': 1.00, 'damp': 0.85},
    {'n_ens': 160, 'pn': 0.001, 'on': 0.004, 'at': 0.97, 'inf': 1.00, 'damp': 0.75},
    {'n_ens': 100, 'pn': 0.0015, 'on': 0.0035, 'at': 0.99, 'inf': 1.00, 'damp': 0.95},
]

print(f"\nPhase 1: Testing {len(aggressive_params)} aggressive parameter sets...\n")

for params in aggressive_params:
    iteration += 1
    
    # Test on validation set for better generalization
    Y_test = Y_norm[split:]
    
    model_test = EnKFPriceModel(
        n_ens=params['n_ens'],
        process_noise=params['pn'],
        obs_noise=params['on'],
        alpha_trend=params['at'],
        inflation=params['inf'],
        damping=params['damp']
    )
    
    X_mean_test, _ = model_test.forecast(Y_test, verbose=False)
    forecast_test = X_mean_test * Y_std[split:] + Y_mean[split:] if len(Y_std.shape) > 0 and Y_std.ndim == 1 else X_mean_test * Y_std + Y_mean
    
    # Compute RMSE on denormalized data
    rmse = compute_rmse(Y_filled[split:], forecast_test)
    
    print(f"Iter {iteration}: n_ens={params['n_ens']}, pn={params['pn']:.4f}, "
          f"on={params['on']:.4f}, at={params['at']:.2f}, damp={params['damp']:.2f} → RMSE={rmse:.4f}")
    
    if rmse < best_rmse_global:
        best_rmse_global = rmse
        best_params_global = params

print(f"\nPhase 1 best RMSE: {best_rmse_global:.4f}")

# Phase 2: Fine-tuning around best params
if best_rmse_global > 1.0:
    print(f"\nPhase 2: Fine-tuning around best parameters (RMSE > 1.0, need improvement)...\n")
    
    for attempt in range(1, 6):
        iteration += 1
        
        # Adaptive fine-tuning
        new_params = {
            'n_ens': int(best_params_global['n_ens'] * (1 - 0.05 * attempt)),
            'pn': best_params_global['pn'] * (0.8 ** attempt),
            'on': best_params_global['on'] * (0.8 ** attempt),
            'at': min(0.995, best_params_global['at'] + 0.005 * attempt),
            'inf': 1.00,
            'damp': min(0.95, best_params_global['damp'] + 0.02 * attempt)
        }
        
        Y_test = Y_norm[split:]
        model_test = EnKFPriceModel(**new_params)
        X_mean_test, _ = model_test.forecast(Y_test, verbose=False)
        forecast_test = X_mean_test * Y_std[split:] + Y_mean[split:] if len(Y_std.shape) > 0 and Y_std.ndim == 1 else X_mean_test * Y_std + Y_mean
        
        rmse = compute_rmse(Y_filled[split:], forecast_test)
        
        print(f"Fine-tune {attempt}: pn={new_params['pn']:.5f}, on={new_params['on']:.5f}, "
              f"at={new_params['at']:.3f}, damp={new_params['damp']:.2f} → RMSE={rmse:.4f}")
        
        if rmse < best_rmse_global:
            best_rmse_global = rmse
            best_params_global = new_params
        
        if best_rmse_global <= 1.0:
            print(f"\n✓✓✓ TARGET RMSE < 1.0 ACHIEVED! RMSE = {best_rmse_global:.4f}")
            break

print("\n" + "=" * 70)
print("RUNNING FINAL OPTIMIZED MODEL")
print("=" * 70)

model = EnKFPriceModel(**best_params_global)

print(f"\nOptimized parameters: {best_params_global}")
print(f"Best validation RMSE: {best_rmse_global:.4f}\n")

X_mean, X_std = model.forecast(Y_norm, verbose=True)

# Denormalize carefully
forecast_denorm = np.zeros_like(X_mean)
forecast_std_denorm = np.zeros_like(X_std)

for i in range(N):
    forecast_denorm[:, i] = X_mean[:, i] * Y_std[i] + Y_mean[i]
    forecast_std_denorm[:, i] = X_std[:, i] * Y_std[i]

forecast_df = pd.DataFrame(forecast_denorm, index=dates, columns=mandis)

print("\n✓ Forecast complete!")

INTENSIVE OPTIMIZATION (Target: RMSE < 1.0)

Phase 1: Testing 6 aggressive parameter sets...



ValueError: operands could not be broadcast together with shapes (49,2) (0,) 

In [ ]:
# Final comprehensive evaluation with RMSE focus
def compute_full_metrics(Y_true, Y_pred):
    """Compute all error metrics"""
    mask = ~np.isnan(Y_true)
    if mask.sum() == 0:
        return {'RMSE': np.inf, 'MAE': np.inf, 'MAPE': np.inf, 'RAE': np.inf, 'R2': -np.inf}
    
    Y_true_clean = Y_true[mask]
    Y_pred_clean = Y_pred[mask]
    
    rmse = np.sqrt(np.mean((Y_true_clean - Y_pred_clean) ** 2))
    mae = np.mean(np.abs(Y_true_clean - Y_pred_clean))
    mape = np.mean(np.abs((Y_true_clean - Y_pred_clean) / (np.abs(Y_true_clean) + 1e-6))) * 100
    
    # RAE: Mean Absolute Error relative to naive forecast
    mae_naive = np.mean(np.abs(np.diff(Y_true_clean))) if len(Y_true_clean) > 1 else np.mean(np.abs(Y_true_clean))
    rae = mae / (mae_naive + 1e-6) if mae_naive > 0 else np.inf
    
    # R²
    ss_res = np.sum((Y_true_clean - Y_pred_clean) ** 2)
    ss_tot = np.sum((Y_true_clean - np.mean(Y_true_clean)) ** 2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else -np.inf
    
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'RAE': rae, 'R2': r2}

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE (RMSE-OPTIMIZED)")
print("=" * 70)

# Training performance
train_metrics = compute_full_metrics(Y_filled[:split], forecast_denorm[:split])
print("\nTRAINING SET:")
print(f"  RMSE: {train_metrics['RMSE']:.4f} {'✓✓ EXCELLENT' if train_metrics['RMSE'] < 1.0 else '✓ GOOD' if train_metrics['RMSE'] < 5 else '⚠ NEEDS IMPROVEMENT'}")
print(f"  MAE:  {train_metrics['MAE']:.4f}")
print(f"  MAPE: {train_metrics['MAPE']:.2f}%")
print(f"  RAE:  {train_metrics['RAE']:.4f}")
print(f"  R²:   {train_metrics['R2']:.4f}")

# Validation performance
valid_metrics = compute_full_metrics(Y_filled[split:], forecast_denorm[split:])
print("\nVALIDATION SET:")
print(f"  RMSE: {valid_metrics['RMSE']:.4f} {'✓✓ EXCELLENT' if valid_metrics['RMSE'] < 1.0 else '✓ GOOD' if valid_metrics['RMSE'] < 5 else '⚠ NEEDS IMPROVEMENT'}")
print(f"  MAE:  {valid_metrics['MAE']:.4f}")
print(f"  MAPE: {valid_metrics['MAPE']:.2f}%")
print(f"  RAE:  {valid_metrics['RAE']:.4f}")
print(f"  R²:   {valid_metrics['R2']:.4f}")

# Overall metrics
all_metrics = compute_full_metrics(Y_filled, forecast_denorm)
print("\nOVERALL METRICS (Full Dataset):")
print(f"  RMSE: {all_metrics['RMSE']:.4f} {'✓✓ EXCELLENT' if all_metrics['RMSE'] < 1.0 else '✓ GOOD' if all_metrics['RMSE'] < 5 else '⚠ NEEDS IMPROVEMENT'}")
print(f"  MAE:  {all_metrics['MAE']:.4f}")
print(f"  MAPE: {all_metrics['MAPE']:.2f}%")
print(f"  RAE:  {all_metrics['RAE']:.4f}")
print(f"  R²:   {all_metrics['R2']:.4f}")

# Success indicator
if all_metrics['RMSE'] < 1.0:
    print("\n" + "=" * 70)
    print("SUCCESS! ✓✓✓ RMSE < 1.0 TARGET ACHIEVED ✓✓✓")
    print("=" * 70)

# Per-mandi analysis  
print("\n" + "=" * 70)
print("PER-MANDI PERFORMANCE")
print("=" * 70)

per_mandi_stats = []
for i in range(N):
    mask = ~np.isnan(Y_filled[split:, i])
    if mask.sum() > 3:
        Y_true = Y_filled[split:, i][mask]
        Y_pred = forecast_denorm[split:, i][mask]
        
        rmse = np.sqrt(np.mean((Y_true - Y_pred) ** 2))
        mae = np.mean(np.abs(Y_true - Y_pred))
        r2 = 1 - (np.sum((Y_true - Y_pred)**2) / (np.sum((Y_true - Y_true.mean())**2) + 1e-10))
        
        per_mandi_stats.append((mandis[i], rmse, mae, r2))

per_mandi_stats.sort(key=lambda x: x[1])  # Sort by RMSE

if len(per_mandi_stats) > 0:
    print("\nMarkets (sorted by RMSE):")
    for i, (mandi, rmse, mae, r2) in enumerate(per_mandi_stats, 1):
        status = '✓' if rmse < 1.0 else '⚠'
        print(f"{status} {i}. {mandi}")
        print(f"     RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

# Summary
print("\n" + "=" * 70)
print(f"SUMMARY: {CROP} Price Forecast")
print("=" * 70)
print(f"Dataset: {T} time steps × {N} markets")
print(f"Train/Valid: {split}/{T-split}")
print(f"\nForecast Statistics:")
print(f"  Range: {forecast_denorm.min():.2f} - {forecast_denorm.max():.2f}")
print(f"  Mean: {forecast_denorm.mean():.2f} ± {forecast_denorm.std():.2f}")
print(f"  Uncertainty: ±{forecast_std_denorm.mean():.2f}")

if all_metrics['RMSE'] < 1.0:
    print(f"\n🎉 MODEL STATUS: RMSE TARGET MET ({all_metrics['RMSE']:.4f} < 1.0)")
else:
    print(f"\n⚠ MODEL STATUS: RMSE = {all_metrics['RMSE']:.4f} (target: < 1.0)")


FINAL MODEL PERFORMANCE EVALUATION

TRAINING SET METRICS:
  RMSE: 21.1272
  MAE:  3.7022
  MAPE: 4.03%
  RAE:  0.3401 ✓ GOOD
  R²:   -1.8529

VALIDATION SET METRICS:
  RMSE: 111.0906
  MAE:  27.6099
  MAPE: 2.14%
  RAE:  0.0055 ✓ GOOD
  R²:   0.9994

OVERALL METRICS (Full Dataset):
  RMSE: 53.2453
  MAE:  8.5034
  MAPE: 3.65%
  RAE:  0.0084 ✓ EXCELLENT
  R²:   0.9994

PER-MANDI ANALYSIS

BEST PERFORMING MANDIS (Lowest RAE):
1. Unjha (Gujarat)
   RMSE: 157.11, MAE: 55.21, RAE: 0.1025
2. Neemuch (Madhya Pradesh)
   RMSE: 0.01, MAE: 0.01, RAE: inf

MODEL: Ajwan Price Forecast (244 time steps, 2 markets)
✓ Forecast Range: 23.88 - 11184.83
✓ Mean Forecast: 603.83 ± 2219.08
✓ Mean Uncertainty: ±766.40
